In [ ]:
!pip install -q transformers accelerate bitsandbytes rapidfuzz pandas

import torch

In [ ]:
# NER & Relation Extraction
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
)

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = "auto",
    quantization_config = quantization_config,
)

system_prompt = """Bạn là một chuyên gia Trích xuất Thông tin Y khoa (Medical NER Expert). Nhiệm vụ của bạn là đọc các đoạn văn bản y khoa tự do và trích xuất các thực thể chính xác từng ký tự.
QUY TẮC - QUÉT TOÀN DIỆN:
- Bắt buộc đọc từ ký tự đầu tiên đến ký tự cuối cùng của văn bản. TUYỆT ĐỐI KHÔNG được bỏ sót bất kỳ một chẩn đoán, triệu chứng, xét nghiệm hay loại thuốc nào.
- text: cụm từ trong input mà hệ thống xác định là một khái niệm y tế
- type: loại khái niệm y tế, bao gồm 1 trong các nhãn như sau:
+ TRIỆU_CHỨNG: Tên triệu chứng bệnh nhân mắc phải
+ TÊN_XÉT_NGHIỆM: Tên xét nghiệm bệnh nhân thực hiện
+ KẾT_QUẢ_XÉT_NGHIỆM: Kết quả xét nghiệm bệnh nhân thực hiện, bao gồm giá trị và đơn vị của xét nghiệm
+ CHẨN_ĐOÁN: Tên chẩn đoán của bác sĩ về bệnh mà bệnh nhân mắc phải
+ THUỐC: Tên thuốc mà bệnh nhân điều trị
- assertions: các mối liên hệ của khái niệm y khoa (ở đây chỉ giới hạn trong CHẨN_ĐOÁN, THUỐC và TRIỆU_CHỨNG) trong bối cảnh văn bản y khoa được cung cấp, được cung cấp dưới dạng 1 list bao gồm các chuỗi thể hiện mối liên hệ này. List này có tối đa 3 phần tử như sau:
+ "isNegated": khái niệm bị phủ định trong văn bản (VD: "không ho")
+ "isFamily": khái niệm có liên quan đến tình trạng của người nhà, họ hàng với bệnh nhân (VD: "bố bệnh nhân xuất hiện trường hợp đau bụng tương tự")
+ "isHistorical": khái niệm có liên quan đến tiền sử bệnh nhân (VD: "có tiền sử hen suyễn")

QUY TẮC TRÍCH XUẤT CỐT LÕI:
1. Bạn CHỈ được phép sinh ra các trường dữ liệu sau: "text", "type", "position", "en_text", "assertions".
2. Trường "text" BẮT BUỘC phải là một chuỗi con (substring) có thật, cắt chính xác từ văn bản gốc. Không dịch, không thêm bớt từ.
3. Trường "en_text" BẮT BUỘC phải có đối với các Type: CHẨN_ĐOÁN, TRIỆU_CHỨNG, THUỐC. Nó là bản dịch tiếng Anh chuẩn y khoa của trường "text".

QUY TẮC PHÂN LOẠI (TYPE):
- CHẨN_ĐOÁN: Các bệnh lý, hội chứng đã được bác sĩ kết luận.
- TRIỆU_CHỨNG: Các dấu hiệu lâm sàng, cảm giác khó chịu mà bệnh nhân gặp phải.
- THUỐC: CHỈ bao gồm tên các loại hóa dược, thuốc Tây y dùng để điều trị. NGHIÊM CẤM gán nhãn THUỐC cho: thực phẩm (đậu), hóa chất sinh hoạt (long não, băng phiến), hoặc các hành động (ví dụ: "không mua thuốc", "tránh sử dụng").
- TÊN_XÉT_NGHIỆM / KẾT_QUẢ_XÉT_NGHIỆM: Tên các chỉ số cận lâm sàng và kết quả tương ứng.

QUY TRÌNH LỌC TỪ (CHỐNG NHIỄU):
KHÔNG trích xuất Động từ chỉ hành động (vd: nhập viện, theo dõi, khám, bảo vệ, hiến máu).
KHÔNG trích xuất Danh từ chỉ trạng thái chung chung (vd: tình trạng, yếu tố, nguy cơ, biểu hiện nhẹ).
TRIỆU CHỨNG phải là dấu hiệu tổn thương thể chất/tinh thần (vd: sốt cao, vàng da, bại não, khó thở).

QUY TẮC NHẬN DIỆN PHỦ ĐỊNH (BẮT BUỘC):
Bất cứ khi nào Triệu chứng, Bệnh, hoặc Thuốc xuất hiện ngay sau các từ: "không", "chưa", "chẳng", "phủ nhận", "loại trừ"... Bạn BẮT BUỘC phải thực hiện 2 thao tác:
Xóa bỏ từ phủ định khỏi trường "text". (vd: "không sốt" -> "text": "sốt").
Gán "isNegated" vào trường "assertions".
Nếu không có từ phủ định, để mảng rỗng [].

QUY TẮC TỌA ĐỘ (POSITION):
- Trả về mảng gồm 2 số nguyên [start_index, end_index] đếm theo ký tự.

QUY TẮC SUY LUẬN NGHIÊM NGẶT (BẮT BUỘC TUÂN THỦ):
1. QUÉT TOÀN DIỆN: Không được bỏ sót bất kỳ thực thể nào xuất hiện trong câu, kể cả khi bệnh nhân "không bị" bệnh đó hay "không có" triệu chứng đó.
2. XỬ LÝ PHỦ ĐỊNH (RẤT QUAN TRỌNG)
Nếu câu chứa:
- không ho
- không sốt
- không khó thở
- chưa đau bụng
- phủ nhận đau ngực

THÌ:
- "text" PHẢI chỉ chứa tên thực thể y tế.
- KHÔNG được giữ lại các từ phủ định như:
  "không", "chưa", "chẳng", "phủ nhận", "không ghi nhận", "âm tính với".

Định nghĩa trường "text":
"text" phải là tên chuẩn (canonical mention) của thực thể y tế.
Không được bao gồm:
- từ phủ định
- trạng từ
- tiền sử
- người nhà
- động từ
"text" chỉ bao gồm tên của:
- bệnh
- triệu chứng
- thuốc
- xét nghiệm
- kết quả xét nghiệm

Ví dụ đúng:
Input:
"không khó thở"

Output:
{
    "text":"khó thở",
    "type":"TRIỆU_CHỨNG",
    "assertions":["isNegated"]
}

Ví dụ SAI:
{
    "text":"không khó thở",
    ...
}
3. DỊCH THUẬT Y KHOA: Đối với các thực thể loại "CHẨN_ĐOÁN" BẮT BUỘC phải tạo thêm một trường "en_text" chứa bản dịch tiếng Anh chuyên ngành tương ứng để phục vụ đối chiếu hệ thống ICD-10.
4. CHỈ trả về một mảng JSON Array hợp lệ. Tuyệt đối không sinh ra bất kỳ văn bản nào nằm ngoài định dạng JSON.
5. "text" luôn phải là tên chuẩn của thực thể.
"text" KHÔNG được chứa:
- không
- chưa
- chẳng
- phủ nhận
- có tiền sử
- gia đình
- cha
- mẹ
- anh trai
- chị gái
- hiện đang
- đang dùng
Những thông tin này chỉ được biểu diễn bằng assertions.
6. TRÍCH XUẤT NGUYÊN BẢN (EXACT MATCH) & CHỐNG ẢO GIÁC:
- Giá trị của trường "text" BẮT BUỘC phải là một chuỗi con (substring) tồn tại chính xác từng ký tự trong văn bản gốc.
- TUYỆT ĐỐI KHÔNG tự ý dịch sang tiếng Anh, KHÔNG sửa lỗi chính tả, KHÔNG thêm bớt từ vào trường "text". (Phần dịch tiếng Anh chỉ được phép ghi vào trường "en_text").
- Độ dài tối đa của "text": Không vượt quá 7 từ. Nếu bạn định trích xuất một đoạn dài hơn (chứa các từ như "không sử dụng", "cần tránh"), hãy dừng lại và chỉ lấy phần danh từ cốt lõi (ví dụ: "đậu tằm", "thuốc giảm đau").
"""

few_shot_examples = [
  {
   "role": "user",
   "content": "Bệnh nhân không ho, có tiền sử trào ngược dạ dày, hiện đang dùng aspirin 81mg."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "ho",
    "type": "TRIỆU_CHỨNG",
    "assertions": ["isNegated"]
  },
  {
    "text": "trào ngược dạ dày",
    "type": "CHẨN_ĐOÁN",
    "assertions": ["isHistorical"]
    "en_text": "gastroesophageal reflux disease"
  },
  {
    "text": "aspirin 81mg",
    "type": "THUỐC",
    "assertions": []
  }
  """
  },

  {
   "role": "user",
   "content": "Bệnh nhân chưa ghi nhận đau ngực, chưa buồn nôn."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "đau ngực",
    "type": "TRIỆU_CHỨNG",
    "assertions": ["isNegated"]
  },
  {
    "text": "buồn nôn",
    "type": "TRIỆU_CHỨNG",
    "assertions": ["isNegated"]
  }
  """
  },

  {
   "role": "user",
   "content": "Bệnh nhân có tiền sử hen phế quản, hiện không còn khó thở."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "hen phế quản",
    "type": "CHẨN_ĐOÁN",
    "assertions": ["isHistorical"]
    "en_text": "asthma"
  },
  {
    "text": "khó thở",
    "type": "TRIỆU_CHỨNG",
    "assertions": ["isNegated"]
  }
  """
  },

  {
   "role": "user",
   "content": "Bệnh nhân phủ nhận khó thở nhưng có ho khan."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "khó thở",
    "type": "TRIỆU_CHỨNG",
    "assertions": ["isNegated"]
  },
  {
    "text": "ho khan",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  }
  """
  },

  {
   "role": "user",
   "content": "Bố bệnh nhân từng bị ung thư phổi. Bệnh nhân nhập viện vì tức ngực dữ dội, không khó thở. Đã cho dùng Paracetamol 500mg giảm đau."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "ung thư phổi",
    "type": "CHẨN_ĐOÁN",
    "assertions": ["isFamily"]
    "en_text": "lung cancer"
  },
  {
    "text": "tức ngực",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "khó thở",
    "type": "TRIỆU_CHỨNG",
    "assertions": [isNegated]
  },
  {
    "text": "Paracetamol 500mg",
    "type": "THUỐC",
    "assertions": []
  }
  """
  },

  {
   "role": "user",
   "content": "Bệnh nhân nam 70 tuổi bị bệnh 1 tuần nay, ho đờm xanh, tức ngực, đau thượng vị, ợ hơi, được chẩn đoán mắc bệnh trào ngược dạ dày - thực quản. Bệnh nhân có tiền sử sử dụng Chlorpheniramine 0.4 MG/ML, Capsaicin 0.38 MG/ML, đã tiến hành tổng phân tích tế bào máu bằng máy lazer (tbm): WBC:14,43; NEUT% (Tỷ lệ % bạch cầu trung tính):76,4; LYPH% (Tỷ lệ bạch cầu lympho):12,8;"
  },
  {
   "role": "assistant",
   "content": """
  {
    "text": "bệnh trào ngược dạ dày - thực quản",
    "type": "CHẨN_ĐOÁN",
    "assertions": []
    "en_text": "gastroesophageal reflux disease"
  },
  {
    "text": "ho đờm xanh",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "tức ngực",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "đau thượng vị",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "ợ hơi",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "TWBC",
    "type": "TÊN_XÉT_NGHIỆM"
  },
  {
    "text": "NEUT% (Tỷ lệ % bạch cầu trung tính)",
    "type": "TÊN_XÉT_NGHIỆM"
  },
  {
    "text": "LYPH% (Tỷ lệ bạch cầu lympho)",
    "type": "TÊN_XÉT_NGHIỆM"
  },
  {
    "text": "14,43",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  },
  {
    "text": "76,4",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  },
  {
    "text": "12,8",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  },
  {
    "text": "Chlorpheniramine 0.4 MG/ML",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "Capsaicin 0.38 MG/ML",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  }
  """
  },

  {
   "role": "user",
   "content": "Danh sách thuốc trước nhập viện chính xác và đầy đủ. 1. amlodipine 10 mg po daily 2. aspirin 81 mg po daily 3. metoprolol succinate xl 50 mg po daily 4. guaifenesin ml po q6h:prn điều trị ho 5. nystatin oral suspension 5 ml po qid:prn điều trị đau nhức 6. acetaminophen 325-650 mg po q6h:prn điều trị sốt đau 7. pravastatin 40 mg po daily 8. docusate sodium 100 mg po bid điều trị táo bón 9. senna 8.6 mg po bid:prn điều trị táo bón 10. clonazepam 0.5 mg po qam:prn điều trị lo âu 11. clonazepam 1.5 mg po qhs điều trị lo âu mất ngủ"
  },
  {
   "role": "assistant",
   "content": """
  {
    "text": "amlodipine 10 mg po daily",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "aspirin 81 mg po daily",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "metoprolol succinate xl 50 mg po daily",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "guaifenesin ml po q6h:prn",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "ho",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "nystatin oral suspension 5 ml po qid:prn",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "đau nhức",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "acetaminophen 325-650 mg po q6h:prn",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "sốt đau",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "pravastatin 40 mg po daily",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "docusate sodium 100 mg po bid",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "táo bón",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "senna 8.6 mg po bid:prn",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "táo bón",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "clonazepam 0.5 mg po qam:prn",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "lo âu",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "clonazepam 1.5 mg po qhs",
    "type": "THUỐC",
    "assertions": ["isHistorical"]
  },
  {
    "text": "lo âu",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  },
  {
    "text": "mất ngủ",
    "type": "TRIỆU_CHỨNG",
    "assertions": []
  }
  """
  },
  {
   "role": "user",
   "content": "HbA1c: 8.2%; Glucose máu lúc đói: 9.5 mmol/L."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "HbA1c",
    "type": "TÊN_XÉT_NGHIỆM"
  },
  {
    "text": "8.2%",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  },
  {
    "text": "Glucose máu lúc đói",
    "type": "TÊN_XÉT_NGHIỆM"
  },
  {
    "text": "9.5 mmol/L",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  }
  """
  },
  {
   "role": "user",
   "content": "Bệnh nhân có tiền sử tăng huyết áp nhưng hiện không còn sử dụng Amlodipine."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "tăng huyết áp",
    "type": "CHẨN_ĐOÁN",
    "assertions": ["isHistorical"]
  },
  {
    "text": "Amlodipine",
    "type": "THUỐC",
    "assertions": ["isNegated"]
  }
  """
  },
  {
   "role": "user",
   "content": "Cha bệnh nhân mắc ung thư đại trực tràng, bệnh nhân không có triệu chứng đau bụng."
  },
  {
    "role": "assistant",
    "content": """
  {
    "text": "ung thư đại trực tràng",
    "type": "CHẨN_ĐOÁN",
    "assertions": ["isFamily"]
  },
  {
    "text": "đau bụng",
    "type": "TRIỆU_CHỨNG",
    "assertions": ["isNegated"]
  }
  """
  },
  {"role": "user", "content": "Bệnh nhân không khó thở, chưa đau bụng. Bác sĩ yêu cầu theo dõi thêm và không nhập viện."},
  {"role": "assistant", "content": """[
    {
        "text": "khó thở",
        "type": "TRIỆU_CHỨNG",
        "assertions": ["isNegated"],
        "en_text": "dyspnea"
    },
    {
        "text": "đau bụng",
        "type": "TRIỆU_CHỨNG",
        "assertions": ["isNegated"],
        "en_text": "abdominal pain"
    }
    ]
   """
  }
]

In [ ]:
import re
import json
import zipfile
import shutil
import pandas as pd
from pathlib import Path
from rapidfuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class MedicalEntityPipeline:
    def __init__(self, model, tokenizer, icd10_path, rxnorm_path, system_prompt, few_shot_examples):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.few_shot_examples = few_shot_examples

        print("Đang nạp cơ sở dữ liệu...")
        self.df_icd10 = pd.read_csv(icd10_path)
        self.df_rxnorm = pd.read_csv(rxnorm_path)

        self.list_icd_names = self.df_icd10['NAME'].fillna("").astype(str).tolist()
        self.list_rx_names = self.df_rxnorm['STR'].fillna("").astype(str).tolist()

        print("Vectorizing...")
        self.vectorizer_icd = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
        self.vectorizer_rx = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))

        self.tfidf_matrix_icd = self.vectorizer_icd.fit_transform(self.list_icd_names)
        self.tfidf_matrix_rx = self.vectorizer_rx.fit_transform(self.list_rx_names)

        print("Hoàn tất khởi tạo Pipeline!")

    def _salvage_json_objects(self, broken_json_string):
        valid_entities = []
        object_blocks = re.findall(r'\{.*?\}', broken_json_string, re.DOTALL)

        for block in object_blocks:
            try:
                entity = json.loads(block)
                if "text" in entity and "type" in entity:
                    valid_entities.append(entity)
            except Exception:
                continue

        return valid_entities

    def _extract_entities(self, input_text, max_retries=3):
        # Module 1: LLM trích xuất NER
        messages = [{"role": "system", "content": self.system_prompt}] + self.few_shot_examples + [{"role": "user", "content": input_text}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        last_cleaned_output = ""

        for attempt in range(max_retries):
            generated_ids = self.model.generate(
                model_inputs.input_ids,
                max_new_tokens = 2048,
                temperature = 0.0,
                do_sample = False,
                top_p = 1.0,
                repetition_penalty = 1.15,
                pad_token_id = self.tokenizer.eos_token_id
            )

            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
            response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

            cleaned_output = re.sub(r"```(?:json)?", "", response).strip()
            cleaned_output = re.sub(r'\}\s*\{', '}, {', cleaned_output)

            last_cleaned_output = cleaned_output

            try:
                return json.loads(cleaned_output)
            except Exception as e:
                print(f"Lỗi Parse JSON (Lần thử {attempt + 1}/{max_retries}): {e}")

        print("Đang kích hoạt giao thức cứu vớt dữ liệu (Data Salvaging)...")
        salvaged_data = self._salvage_json_objects(last_cleaned_output)

        if salvaged_data:
            print(f"Đã cứu vớt thành công {len(salvaged_data)} thực thể hợp lệ từ file lỗi.")
            return salvaged_data
        return []

    def _get_best_match(self, query, vectorizer, tfidf_matrix, threshold, dict_names):
        if not query: return None, 0.0

        query_vec = vectorizer.transform([query])
        cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

        top_10_indices = cosine_similarities.argsort()[-10:][::-1]

        best_idx = None
        best_fuzz_score = 0.0

        for idx in top_10_indices:
            candidate_str = dict_names[idx]
            fuzz_score = fuzz.WRatio(query.lower(), str(candidate_str).lower())
            if fuzz_score > best_fuzz_score:
                best_fuzz_score = fuzz_score
                best_idx = idx

        if best_fuzz_score >= threshold:
            return best_idx, best_fuzz_score
        return None, best_fuzz_score

    def _deduplicate_entities(self, entities):
        if not entities: return []

        sorted_entities = sorted(entities, key=lambda x: x["position"][0])
        deduped = [sorted_entities[0]]

        for current in sorted_entities[1:]:
            prev = deduped[-1]
            prev_start, prev_end = prev["position"]
            curr_start, curr_end = current["position"]

            if curr_start < prev_end:
                if (curr_end - curr_start) > (prev_end - prev_start):
                    deduped[-1] = current
            else:
                deduped.append(current)

        return deduped

    def _post_process_entities(self, original_text, entities):
        BLACKLIST_WORDS = [
            "không", "chưa", "mua", "uống", "tránh", "sử dụng", "thực phẩm",
            "băng phiến", "long não", "đậu", "hay", "nhập viện", "theo dõi",
            "nguy cơ", "yếu tố", "tình trạng", "bảo vệ", "sản sinh", "mẹ",
            "chỉ định", "nguy hiểm", "thuốc nam", "thuốc đông y"
        ]

        VALID_TYPES = ["TRIỆU_CHỨNG", "CHẨN_ĐOÁN", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"]

        search_text = original_text.lower()
        cleaned_entities = []

        for entity in entities:
            raw_text = entity.get("text", "").strip()
            e_type = entity.get("type", "").strip().upper()
            en_text = entity.get("en_text", "").strip()

            best_type_match = process.extractOne(e_type, VALID_TYPES, scorer=fuzz.WRatio)
            if best_type_match and best_type_match[1] > 80:
                e_type = best_type_match[0]
            else:
                continue

            if not raw_text or len(raw_text.split()) > 6:
                continue
            if any(word in raw_text.lower() for word in BLACKLIST_WORDS):
                continue

            mapped_code = None
            search_query = en_text if en_text else raw_text

            if e_type in ["CHẨN_ĐOÁN", "TRIỆU_CHỨNG"]:
                best_idx, score = self._get_best_match(search_query, self.vectorizer_icd, self.tfidf_matrix_icd, 90, self.list_icd_names)

                if best_idx is not None:
                    mapped_code = self.df_icd10.iloc[best_idx]['CODE']
                    e_type = "TRIỆU_CHỨNG" if str(mapped_code).startswith('R') else "CHẨN_ĐOÁN"
                else:
                    e_type = "TRIỆU_CHỨNG"

            elif e_type == "THUỐC":
                best_idx, score = self._get_best_match(search_query, self.vectorizer_rx, self.tfidf_matrix_rx, 85, self.list_rx_names)

                if best_idx is not None:
                    mapped_code = str(self.df_rxnorm.iloc[best_idx]['RXCUI'])
                else:
                    mapped_code = None

            entity["type"] = e_type

            if e_type in ["CHẨN_ĐOÁN", "THUỐC"]:
                entity["candidates"] = [mapped_code] if mapped_code else []
            else:
                if "candidates" in entity:
                    del entity["candidates"]

            pattern = re.escape(raw_text.lower())
            match = re.search(pattern, search_text, re.IGNORECASE)

            if match:
                start_idx, end_idx = match.start(), match.end()
            else:
                extracted = process.extractOne(
                    raw_text.lower(),
                    [search_text],
                    scorer=fuzz.partial_ratio,
                    score_cutoff = 85
                )

                if extracted:
                    words = raw_text.split()
                    fallback_match = None
                    while len(words) > 1 and not fallback_match:
                        words = words[1:]
                        fallback_match = re.search(re.escape(" ".join(words)), search_text, re.IGNORECASE)

                    if fallback_match:
                        start_idx, end_idx = fallback_match.start(), fallback_match.end()
                    else:
                        continue
                else:
                    continue

            entity["position"] = [start_idx, end_idx]
            search_text = search_text[:start_idx] + " " * (end_idx - start_idx) + search_text[end_idx:]

            context_window = original_text[max(0, start_idx - 25):start_idx].lower()

            negation_triggers = ["không", "chưa", "chẳng", "phủ nhận", "loại trừ"]
            family_triggers = ["bố", "mẹ", "anh", "chị", "em", "gia đình"]
            historical_triggers = ["tiền sử", "trước đây", "đã từng"]

            assertions = []
            if any(trigger in context_window for trigger in negation_triggers):
                assertions.append("isNegated")
            if any(trigger in context_window for trigger in family_triggers):
                assertions.append("isFamily")
            if any(trigger in context_window for trigger in historical_triggers):
                assertions.append("isHistorical")

            entity["assertions"] = list(set(entity.get("assertions", []) + assertions))

            if "en_text" in entity:
                del entity["en_text"]

            cleaned_entities.append(entity)

        return self._deduplicate_entities(cleaned_entities)

    def process_single_text(self, text):
        # Module 3: Điều phối luồng dữ liệu + Chunking
        if not text: return []

        sentences = re.split(r'(?<=[.!?\n])\s+', text.strip())

        chunks = []
        current_chunk = ""
        max_chunk_len = 600
        overlap_len = 100

        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= max_chunk_len:
                current_chunk += sentence + " "
            else:
                chunks.append(current_chunk.strip())
                overlap_text = current_chunk[-overlap_len:]
                safe_overlap_idx = overlap_text.find(' ')
                if safe_overlap_idx != -1:
                    overlap_text = overlap_text[safe_overlap_idx:]
                current_chunk = overlap_text + sentence + " "

        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        all_raw_entities = []

        for i, chunk in enumerate(chunks):
            if len(chunk) < 15:
                continue

            chunk_entities = self._extract_entities(chunk)

            if chunk_entities:
                all_raw_entities.extend(chunk_entities)

        if not all_raw_entities:
            return []

        final_entities = self._post_process_entities(text, all_raw_entities)

        return final_entities

    def process_zip_batch(self, input_zip_path, output_zip_path):
        # Module 4: Xử lý ZIP nguyên bản
        temp_input = Path("temp_inputs")
        temp_output = Path("temp_outputs")

        if temp_input.exists(): shutil.rmtree(temp_input)
        if temp_output.exists(): shutil.rmtree(temp_output)

        temp_input.mkdir()
        temp_output.mkdir()

        print("1. Đang giải nén file...")
        with zipfile.ZipFile(input_zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_input)

        txt_files = list(temp_input.rglob("*.txt"))
        print(f"2. Đang phân tích {len(txt_files)} hồ sơ bệnh án...")

        for file_path in txt_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read().strip()

            if text:
                final_entities = self.process_single_text(text)
                out_path = temp_output / f"{file_path.stem}.json"
                with open(out_path, 'w', encoding='utf-8') as f:
                    json.dump(final_entities, f, ensure_ascii=False, indent=4)

        print("\n3. Đang đóng gói kết quả...")
        shutil.make_archive(output_zip_path.replace(".zip", ""), 'zip', temp_output)

        shutil.rmtree(temp_input)
        shutil.rmtree(temp_output)
        print(f"Hoàn tất! Đã lưu file: {output_zip_path}")

    def process_folder_batch(self, input_folder_path, output_zip_path):
        # Module 5: Checkpointing
        temp_output = Path("temp_outputs")

        if not temp_output.exists():
            temp_output.mkdir()

        input_dir = Path(input_folder_path)
        txt_files = list(input_dir.rglob("*.txt"))

        if len(txt_files) == 0:
            print("Lỗi: Không tìm thấy file .txt nào!")
            return

        print(f"Bắt đầu quét {len(txt_files)} hồ sơ bệnh án...")

        for file_path in txt_files:
            out_path = temp_output / f"{file_path.stem}.json"

            if out_path.exists():
                print(f"Bỏ qua {file_path.name} (Đã xử lý trước đó)")
                continue

            print(f"Đang phân tích: {file_path.name}...")
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read().strip()

            if text:
                final_entities = self.process_single_text(text)
                with open(out_path, 'w', encoding='utf-8') as f:
                    json.dump(final_entities, f, ensure_ascii=False, indent=4)

        print("\nĐang tạo bản sao lưu ZIP...")
        shutil.make_archive(output_zip_path.replace(".zip", ""), 'zip', temp_output)

        print(f"Hoàn tất an toàn!")
        print(f"Dữ liệu thô: temp_outputs")
        print(f"Dữ liệu nén: {output_zip_path}")

In [ ]:
DATA_DIR = "/content/input"
WORKING_DIR = "/content/"

icd_path = "/content/icd10.csv"
rxnorm_path = "/content/rxnorm_drugs_clean.csv"
input_folder = "/content/input"
output_zip = WORKING_DIR + "output.zip"

pipeline = MedicalEntityPipeline(
    model = model,
    tokenizer = tokenizer,
    icd10_path = icd_path,
    rxnorm_path = rxnorm_path,
    system_prompt = system_prompt,
    few_shot_examples = few_shot_examples
)

pipeline.process_folder_batch(input_folder, output_zip)